# Clases y objetos · Estructuras jerárquicas

Árboles binarios de búsqueda, montículos, colas de prioridad y tries implementados como clases. En un árbol binario de búsqueda buscar, insertar y borrar cuestan $O(h)$, con $h$ la altura: $O(\log n)$ si está equilibrado y $O(n)$ en el peor caso. En un montículo `push` y `pop` cuestan $O(\log n)$ y `heapify`, $O(n)$; en un trie las operaciones cuestan $O(L)$, con $L$ la longitud de la palabra.

## 1. Árbol binario de búsqueda: insertar y buscar

Cada nodo tiene un valor y dos hijos; los valores del subárbol izquierdo son menores y los del derecho mayores (sin repetidos). Definir `ABB` con `insertar`, `buscar`, `__contains__` y `__len__`. Comprobar contra `set`.

In [1]:
import random


class NodoArbol:
    def __init__(self, valor):
        self.valor = valor
        self.izq = self.der = None


class ABB:
    def __init__(self, valores=()):
        self.raiz = None
        self.n = 0
        for v in valores:
            self.insertar(v)

    def insertar(self, valor):
        self.raiz = self._insertar(self.raiz, valor)

    def _insertar(self, nodo, valor):
        if nodo is None:
            self.n += 1
            return NodoArbol(valor)
        if valor < nodo.valor:
            nodo.izq = self._insertar(nodo.izq, valor)
        elif valor > nodo.valor:
            nodo.der = self._insertar(nodo.der, valor)
        return nodo

    def buscar(self, valor):
        nodo = self.raiz
        while nodo and nodo.valor != valor:
            nodo = nodo.izq if valor < nodo.valor else nodo.der
        return nodo is not None

    def __contains__(self, valor):
        return self.buscar(valor)

    def __len__(self):
        return self.n


arbol = ABB([50, 30, 70, 20, 40, 60, 80, 35])
arbol.insertar(30)
print(len(arbol), 35 in arbol, 36 in arbol)

random.seed(1)
for _ in range(100):
    valores = [random.randint(0, 100) for _ in range(50)]
    arbol = ABB(valores)
    assert len(arbol) == len(set(valores))
    assert all((x in arbol) == (x in valores) for x in range(-5, 106))
print("Comprobación correcta")

8 True False
Comprobación correcta


## 2. Recorridos

Ampliar la clase (heredando de la anterior) con los recorridos en profundidad `inorden` (izquierda, raíz, derecha), `preorden` (raíz, izquierda, derecha) y `postorden` (izquierda, derecha, raíz), y el recorrido `por_niveles` (en anchura, con una cola), que devuelve una lista por nivel. El inorden de un ABB sale ordenado.

In [2]:
from collections import deque


class ABB(ABB):
    def _inorden(self, nodo):
        if nodo:
            yield from self._inorden(nodo.izq)
            yield nodo.valor
            yield from self._inorden(nodo.der)

    def _preorden(self, nodo):
        if nodo:
            yield nodo.valor
            yield from self._preorden(nodo.izq)
            yield from self._preorden(nodo.der)

    def _postorden(self, nodo):
        if nodo:
            yield from self._postorden(nodo.izq)
            yield from self._postorden(nodo.der)
            yield nodo.valor

    def inorden(self):
        return list(self._inorden(self.raiz))

    def preorden(self):
        return list(self._preorden(self.raiz))

    def postorden(self):
        return list(self._postorden(self.raiz))

    def por_niveles(self):
        niveles, cola = [], deque([self.raiz] if self.raiz else [])
        while cola:
            nivel = []
            for _ in range(len(cola)):
                nodo = cola.popleft()
                nivel.append(nodo.valor)
                cola.extend(hijo for hijo in (nodo.izq, nodo.der) if hijo)
            niveles.append(nivel)
        return niveles

    def __iter__(self):
        return self._inorden(self.raiz)


arbol = ABB([50, 30, 70, 20, 40, 60, 80, 35])
print("Inorden:   ", arbol.inorden())
print("Preorden:  ", arbol.preorden())
print("Postorden: ", arbol.postorden())
print("Por niveles:", arbol.por_niveles())

random.seed(2)
for _ in range(100):
    valores = random.sample(range(1000), random.randint(0, 60))
    arbol = ABB(valores)
    assert arbol.inorden() == list(arbol) == sorted(valores)
    # insertar en preorden reconstruye el mismo árbol
    assert ABB(arbol.preorden()).por_niveles() == arbol.por_niveles()
    assert sorted(arbol.postorden()) == sorted(valores)
    if valores:
        assert arbol.postorden()[-1] == arbol.preorden()[0] == arbol.por_niveles()[0][0] == valores[0]
print("Comprobación correcta")

Inorden:    [20, 30, 35, 40, 50, 60, 70, 80]
Preorden:   [50, 30, 20, 40, 35, 70, 60, 80]
Postorden:  [20, 35, 40, 30, 60, 80, 70, 50]
Por niveles: [[50], [30, 70], [20, 40, 60, 80], [35]]
Comprobación correcta


## 3. Mínimo, máximo y altura

El mínimo está bajando siempre a la izquierda y el máximo a la derecha. La altura es el número de niveles (el árbol vacío tiene altura 0). Comparar la altura de un árbol con claves insertadas en orden (degenera en una lista) con la de uno con claves en orden aleatorio.

In [3]:
class ABB(ABB):
    def minimo(self):
        if self.raiz is None:
            raise ValueError("árbol vacío")
        nodo = self.raiz
        while nodo.izq:
            nodo = nodo.izq
        return nodo.valor

    def maximo(self):
        if self.raiz is None:
            raise ValueError("árbol vacío")
        nodo = self.raiz
        while nodo.der:
            nodo = nodo.der
        return nodo.valor

    def altura(self):
        def altura(nodo):
            return 0 if nodo is None else 1 + max(altura(nodo.izq), altura(nodo.der))
        return altura(self.raiz)


arbol = ABB([50, 30, 70, 20, 40, 60, 80, 35])
print(arbol.minimo(), arbol.maximo(), arbol.altura())
try:
    ABB().minimo()
except ValueError as e:
    print("Error:", e)

random.seed(3)
claves = list(range(500))
print("En orden:", ABB(claves).altura())
random.shuffle(claves)
print("Aleatorio:", ABB(claves).altura())

for _ in range(100):
    valores = random.sample(range(1000), random.randint(1, 60))
    arbol = ABB(valores)
    assert (arbol.minimo(), arbol.maximo()) == (min(valores), max(valores))
    assert arbol.altura() == len(arbol.por_niveles())
print("Comprobación correcta")

20 80 4
Error: árbol vacío
En orden: 500
Aleatorio: 20
Comprobación correcta


## 4. Borrar

Borrar un valor tiene tres casos: si el nodo es una hoja se quita; si tiene un solo hijo, el hijo ocupa su lugar; si tiene dos, se copia en él su sucesor (el mínimo del subárbol derecho) y se borra el sucesor, que tiene a lo sumo un hijo. Devolver si el valor estaba.

In [4]:
class ABB(ABB):
    def borrar(self, valor):
        antes = self.n
        self.raiz = self._borrar(self.raiz, valor)
        return self.n < antes

    def _borrar(self, nodo, valor):
        if nodo is None:
            return None
        if valor < nodo.valor:
            nodo.izq = self._borrar(nodo.izq, valor)
        elif valor > nodo.valor:
            nodo.der = self._borrar(nodo.der, valor)
        elif nodo.izq is None or nodo.der is None:  # hoja o un hijo
            self.n -= 1
            return nodo.izq or nodo.der
        else:  # dos hijos
            sucesor = nodo.der
            while sucesor.izq:
                sucesor = sucesor.izq
            nodo.valor = sucesor.valor
            nodo.der = self._borrar(nodo.der, sucesor.valor)
        return nodo


arbol = ABB([50, 30, 70, 20, 40, 60, 80, 35])
print(arbol.por_niveles())
for valor, caso in [(20, "hoja"), (40, "un hijo"), (50, "dos hijos"), (99, "no existe")]:
    print(f"Borrar {valor} ({caso}): {arbol.borrar(valor)} -> {arbol.por_niveles()}")

random.seed(4)
for _ in range(200):
    valores = set(random.sample(range(100), 40))
    arbol = ABB(valores)
    for x in random.sample(range(100), 60):
        assert arbol.borrar(x) == (x in valores)
        valores.discard(x)
        assert arbol.inorden() == sorted(valores) and len(arbol) == len(valores)
print("Comprobación correcta")

[[50], [30, 70], [20, 40, 60, 80], [35]]
Borrar 20 (hoja): True -> [[50], [30, 70], [40, 60, 80], [35]]
Borrar 40 (un hijo): True -> [[50], [30, 70], [35, 60, 80]]
Borrar 50 (dos hijos): True -> [[60], [30, 70], [35, 80]]
Borrar 99 (no existe): False -> [[60], [30, 70], [35, 80]]
Comprobación correcta


## 5. Validar un árbol de búsqueda

Comprobar si un árbol construido a mano cumple la propiedad de búsqueda. No basta con comparar cada nodo con sus hijos: todo el subárbol izquierdo debe ser menor y todo el derecho mayor, así que se pasa hacia abajo el intervalo $(\min, \max)$ permitido.

In [5]:
def es_abb(nodo, minimo=float("-inf"), maximo=float("inf")):
    if nodo is None:
        return True
    if not minimo < nodo.valor < maximo:
        return False
    return es_abb(nodo.izq, minimo, nodo.valor) and es_abb(nodo.der, nodo.valor, maximo)


def es_abb_ingenuo(nodo):
    if nodo is None:
        return True
    if (nodo.izq and nodo.izq.valor >= nodo.valor) or (nodo.der and nodo.der.valor <= nodo.valor):
        return False
    return es_abb_ingenuo(nodo.izq) and es_abb_ingenuo(nodo.der)


#     10
#    /  \
#   5    20
#    \
#     15   <- mayor que 10 en el subárbol izquierdo
raiz = NodoArbol(10)
raiz.izq, raiz.der = NodoArbol(5), NodoArbol(20)
raiz.izq.der = NodoArbol(15)
print("Ingenuo:", es_abb_ingenuo(raiz), "| Correcto:", es_abb(raiz))

random.seed(5)
for _ in range(200):
    arbol = ABB(random.sample(range(100), 30))
    for x in random.sample(range(100), 10):
        arbol.borrar(x)
    assert es_abb(arbol.raiz)
    if arbol.raiz and arbol.raiz.izq:
        arbol.raiz.izq.valor, arbol.raiz.valor = arbol.raiz.valor, arbol.raiz.izq.valor
        assert not es_abb(arbol.raiz)
print("Comprobación correcta")

Ingenuo: True | Correcto: False
Comprobación correcta


## 6. Montículo binario mínimo

Montículo sobre una lista: el padre de la posición $i$ está en $\lfloor (i-1)/2 \rfloor$ y sus hijos en $2i+1$ y $2i+2$, y cada padre es menor o igual que sus hijos, así que el mínimo está en la posición 0. `push` añade al final y hace subir el elemento; `pop` saca la raíz, pone el último en su lugar y lo hace bajar. Comprobar contra `heapq`.

In [6]:
import heapq


class MonticuloMin:
    def __init__(self):
        self.datos = []

    def push(self, valor):
        self.datos.append(valor)
        self._subir(len(self.datos) - 1)

    def pop(self):
        if not self.datos:
            raise IndexError("montículo vacío")
        ultimo = self.datos.pop()
        if not self.datos:
            return ultimo
        raiz, self.datos[0] = self.datos[0], ultimo
        self._bajar(0)
        return raiz

    def cima(self):
        if not self.datos:
            raise IndexError("montículo vacío")
        return self.datos[0]

    def _subir(self, i):
        d = self.datos
        while i > 0 and d[i] < d[(i - 1) // 2]:
            padre = (i - 1) // 2
            d[i], d[padre] = d[padre], d[i]
            i = padre

    def _bajar(self, i):
        d, n = self.datos, len(self.datos)
        while True:
            menor = i
            for hijo in (2 * i + 1, 2 * i + 2):
                if hijo < n and d[hijo] < d[menor]:
                    menor = hijo
            if menor == i:
                return
            d[i], d[menor] = d[menor], d[i]
            i = menor

    def es_valido(self):
        d = self.datos
        return all(not d[i] < d[(i - 1) // 2] for i in range(1, len(d)))

    def __len__(self):
        return len(self.datos)

    def __repr__(self):
        return f"MonticuloMin({self.datos})"


m = MonticuloMin()
for x in [5, 3, 8, 1, 9, 2]:
    m.push(x)
print(m, m.cima())
print([m.pop() for _ in range(len(m))])
try:
    m.pop()
except IndexError as e:
    print("Error:", e)

random.seed(6)
m, modelo = MonticuloMin(), []
for _ in range(20_000):
    if random.random() < 0.55:
        x = random.randint(0, 1000)
        m.push(x)
        heapq.heappush(modelo, x)
    elif modelo:
        assert m.pop() == heapq.heappop(modelo)
    assert len(m) == len(modelo) and (not modelo or m.cima() == modelo[0])
assert m.es_valido()
print("Comprobación correcta")

MonticuloMin([1, 3, 2, 5, 9, 8]) 1
[1, 2, 3, 5, 8, 9]
Error: montículo vacío
Comprobación correcta


## 7. `heapify` y heap sort

Construir el montículo de una lista en $O(n)$ haciendo bajar cada nodo interno, desde el último hasta la raíz (método de Floyd), en lugar de hacer $n$ `push` ($O(n \log n)$). Usarlo para ordenar sacando el mínimo $n$ veces: heap sort, $O(n \log n)$. Comprobar contra `sorted` y `heapq.heapify`.

In [7]:
class MonticuloMin(MonticuloMin):
    @classmethod
    def heapify(cls, valores):
        m = cls()
        m.datos = list(valores)
        for i in range(len(m.datos) // 2 - 1, -1, -1):
            m._bajar(i)
        return m


def heap_sort(valores):
    m = MonticuloMin.heapify(valores)
    return [m.pop() for _ in range(len(m))]


datos = [9, 4, 7, 1, 8, 2, 6, 3, 5]
print(MonticuloMin.heapify(datos), heap_sort(datos))

random.seed(7)
for _ in range(300):
    valores = [random.randint(-50, 50) for _ in range(random.randint(0, 80))]
    m = MonticuloMin.heapify(valores)
    copia = valores[:]
    heapq.heapify(copia)
    assert m.es_valido() and len(m) == len(copia)
    if valores:
        assert m.cima() == copia[0]
    assert heap_sort(valores) == sorted(valores)
print("Comprobación correcta")

MonticuloMin([1, 3, 2, 4, 8, 7, 6, 9, 5]) [1, 2, 3, 4, 5, 6, 7, 8, 9]


Comprobación correcta


## 8. Cola de prioridad

Cola de prioridad sobre el montículo: se guardan tuplas `(prioridad, orden, elemento)`, donde el contador `orden` desempata para que, a igual prioridad, salga antes el que llegó antes (y para no tener que comparar nunca los elementos). Simular la sala de urgencias de un hospital (prioridad 1 la más grave).

In [8]:
from itertools import count


class ColaPrioridad:
    def __init__(self):
        self.monticulo = MonticuloMin()
        self.contador = count()

    def insertar(self, elemento, prioridad):
        self.monticulo.push((prioridad, next(self.contador), elemento))

    def extraer(self):
        prioridad, _, elemento = self.monticulo.pop()
        return elemento, prioridad

    def __len__(self):
        return len(self.monticulo)


urgencias = ColaPrioridad()
for paciente, prioridad in [("Ana", 3), ("Luis", 1), ("Marta", 2), ("Pedro", 3), ("Carmen", 1), ("Iván", 5)]:
    urgencias.insertar({"nombre": paciente}, prioridad)
while urgencias:
    paciente, prioridad = urgencias.extraer()
    print(f"Prioridad {prioridad}: {paciente['nombre']}")

random.seed(8)
for _ in range(200):
    llegadas = [({"id": i}, random.randint(1, 5)) for i in range(random.randint(0, 40))]
    cola = ColaPrioridad()
    for elemento, prioridad in llegadas:
        cola.insertar(elemento, prioridad)
    # sorted es estable: a igual prioridad respeta el orden de llegada
    assert [cola.extraer() for _ in range(len(cola))] == sorted(llegadas, key=lambda par: par[1])
print("Comprobación correcta")

Prioridad 1: Luis


Prioridad 1: Carmen
Prioridad 2: Marta
Prioridad 3: Ana
Prioridad 3: Pedro
Prioridad 5: Iván
Comprobación correcta


## 9. Trie

Árbol de prefijos: cada nodo tiene un diccionario de hijos (uno por letra) y una marca de fin de palabra. Implementar `insertar`, `buscar` (palabra completa), `empieza_por` (si alguna palabra tiene ese prefijo) y `__len__`.

In [9]:
class NodoTrie:
    def __init__(self):
        self.hijos = {}
        self.fin = False


class Trie:
    def __init__(self, palabras=()):
        self.raiz = NodoTrie()
        self.n = 0
        for palabra in palabras:
            self.insertar(palabra)

    def insertar(self, palabra):
        nodo = self.raiz
        for letra in palabra:
            nodo = nodo.hijos.setdefault(letra, NodoTrie())
        if not nodo.fin:
            nodo.fin = True
            self.n += 1

    def _nodo(self, prefijo):
        nodo = self.raiz
        for letra in prefijo:
            nodo = nodo.hijos.get(letra)
            if nodo is None:
                return None
        return nodo

    def buscar(self, palabra):
        nodo = self._nodo(palabra)
        return nodo is not None and nodo.fin

    def __contains__(self, palabra):
        return self.buscar(palabra)

    def empieza_por(self, prefijo):
        return self._nodo(prefijo) is not None

    def __len__(self):
        return self.n


palabras = ["casa", "casco", "caso", "cosa", "coser", "cama", "camino", "camión", "sol", "solo", "soltar", "casa"]
trie = Trie(palabras)
print(len(trie), "caso" in trie, "cas" in trie, trie.empieza_por("cas"), trie.empieza_por("cx"))

random.seed(9)
for _ in range(100):
    palabras = ["".join(random.choices("abc", k=random.randint(1, 5))) for _ in range(30)]
    trie = Trie(palabras)
    assert len(trie) == len(set(palabras))
    for _ in range(50):
        p = "".join(random.choices("abc", k=random.randint(0, 5)))
        assert (p in trie) == (p in palabras)
        assert trie.empieza_por(p) == any(w.startswith(p) for w in palabras)
print("Comprobación correcta")

11 True False True False


Comprobación correcta


## 10. Autocompletar

Ampliar el trie con `autocompletar(prefijo)`, que devuelve en orden alfabético las palabras que empiezan por el prefijo (se baja hasta su nodo y se recorre el subárbol en profundidad visitando los hijos ordenados), `contar_prefijo` y `prefijo_comun`, el prefijo más largo común a todas las palabras.

In [10]:
import os


class Trie(Trie):
    def autocompletar(self, prefijo, limite=None):
        nodo = self._nodo(prefijo)
        resultado = []
        if nodo:
            self._recoger(nodo, prefijo, resultado, limite)
        return resultado

    def _recoger(self, nodo, palabra, resultado, limite):
        if limite is not None and len(resultado) >= limite:
            return
        if nodo.fin:
            resultado.append(palabra)
        for letra in sorted(nodo.hijos):
            self._recoger(nodo.hijos[letra], palabra + letra, resultado, limite)

    def contar_prefijo(self, prefijo):
        return len(self.autocompletar(prefijo))

    def prefijo_comun(self):
        nodo, prefijo = self.raiz, ""
        while len(nodo.hijos) == 1 and not nodo.fin:
            letra, nodo = next(iter(nodo.hijos.items()))
            prefijo += letra
        return prefijo


trie = Trie(["casa", "casco", "caso", "cosa", "coser", "cama", "camino", "camión", "sol", "solo", "soltar"])
for prefijo in ["ca", "cas", "co", "sol", "x"]:
    print(f"{prefijo!r:6} {trie.contar_prefijo(prefijo)} {trie.autocompletar(prefijo)}")
print(trie.autocompletar("", limite=4))
print(repr(trie.prefijo_comun()), repr(Trie(["programa", "progreso", "prohibir"]).prefijo_comun()))

random.seed(10)
for _ in range(100):
    palabras = ["".join(random.choices("abc", k=random.randint(1, 6))) for _ in range(random.randint(1, 30))]
    trie = Trie(palabras)
    for _ in range(20):
        p = "".join(random.choices("abc", k=random.randint(0, 3)))
        assert trie.autocompletar(p) == sorted({w for w in palabras if w.startswith(p)})
    assert trie.prefijo_comun() == os.path.commonprefix(palabras)
print("Comprobación correcta")

'ca'   6 ['cama', 'camino', 'camión', 'casa', 'casco', 'caso']
'cas'  3 ['casa', 'casco', 'caso']
'co'   2 ['cosa', 'coser']
'sol'  3 ['sol', 'solo', 'soltar']
'x'    0 []
['cama', 'camino', 'camión', 'casa']
'' 'pro'
Comprobación correcta


## 11. Top-k palabras más frecuentes

Contar las palabras de un texto con un diccionario y quedarse con las $k$ más frecuentes usando un montículo mínimo de tamaño $k$ cuya cima es la peor de las guardadas: si llega una mejor, la sustituye. Cuesta $O(n \log k)$ en lugar del $O(n \log n)$ de ordenarlas todas. A igual frecuencia gana la palabra alfabéticamente menor. Comprobar contra `Counter` y `sorted`.

In [11]:
import re
from collections import Counter


class Entrada:
    def __init__(self, palabra, frecuencia):
        self.palabra, self.frecuencia = palabra, frecuencia

    def __lt__(self, otra):  # self es peor que otra
        return (self.frecuencia, otra.palabra) < (otra.frecuencia, self.palabra)


def mejores_k(pares, k):
    monticulo = MonticuloMin()
    for palabra, frecuencia in pares:
        entrada = Entrada(palabra, frecuencia)
        if len(monticulo) < k:
            monticulo.push(entrada)
        elif monticulo.cima() < entrada:
            monticulo.pop()
            monticulo.push(entrada)
    peores_primero = [monticulo.pop() for _ in range(len(monticulo))]
    return [(e.palabra, e.frecuencia) for e in reversed(peores_primero)]


def top_k(texto, k):
    frecuencias = {}
    for palabra in re.findall(r"\w+", texto.lower()):
        frecuencias[palabra] = frecuencias.get(palabra, 0) + 1
    return mejores_k(frecuencias.items(), k)


texto = """En la isla el viento sopla del norte casi todo el año. El viento trae nubes,
las nubes dejan agua en el monte y el agua baja por el barranco hasta el mar.
En verano el viento para, el mar se calma y la isla se llena de gente."""
for palabra, frecuencia in top_k(texto, 5):
    print(f"{palabra:8} {frecuencia}")


def top_k_referencia(texto, k):
    return sorted(Counter(re.findall(r"\w+", texto.lower())).items(), key=lambda par: (-par[1], par[0]))[:k]


random.seed(11)
for _ in range(300):
    texto_aleatorio = " ".join(random.choices(["sol", "mar", "luna", "río", "monte", "nube", "isla", "viento"],
                                              k=random.randint(0, 60)))
    k = random.randint(1, 9)
    assert top_k(texto_aleatorio, k) == top_k_referencia(texto_aleatorio, k)
print("Comprobación correcta")

el       9
en       3
viento   3
agua     2
isla     2
Comprobación correcta


## 12. Sugerencias de un buscador

Combinar el trie y el montículo: un buscador registra cada búsqueda en un trie y guarda cuántas veces se ha hecho; al escribir un prefijo sugiere las $k$ búsquedas más frecuentes que empiezan por él (el trie da las candidatas y `mejores_k` elige las mejores).

In [12]:
class Buscador(Trie):
    def __init__(self):
        super().__init__()
        self.veces = {}

    def registrar(self, busqueda):
        self.insertar(busqueda)
        self.veces[busqueda] = self.veces.get(busqueda, 0) + 1

    def sugerir(self, prefijo, k=3):
        return mejores_k(((p, self.veces[p]) for p in self.autocompletar(prefijo)), k)


random.seed(12)
populares = {"playa": 30, "playa de las canteras": 12, "plátano": 20, "plaza": 8, "pan": 15,
             "papas arrugadas": 25, "parque": 5, "mojo": 18, "montaña": 9, "mapa": 4}
buscador = Buscador()
for busqueda in random.choices(list(populares), weights=list(populares.values()), k=500):
    buscador.registrar(busqueda)

for prefijo in ["p", "pla", "pa", "mo", "z"]:
    print(f"{prefijo!r:6} {buscador.sugerir(prefijo)}")

for prefijo in ["", "p", "pl", "pa", "m", "ma", "x"]:
    for k in range(1, 6):
        esperado = sorted(((p, f) for p, f in buscador.veces.items() if p.startswith(prefijo)),
                          key=lambda par: (-par[1], par[0]))[:k]
        assert buscador.sugerir(prefijo, k) == esperado
print("Comprobación correcta")

'p'    [('playa', 90), ('papas arrugadas', 88), ('plátano', 88)]
'pla'  [('playa', 90), ('playa de las canteras', 44), ('plaza', 23)]
'pa'   [('papas arrugadas', 88), ('pan', 50), ('parque', 13)]
'mo'   [('mojo', 59), ('montaña', 35)]
'z'    []
Comprobación correcta
